### Four in a Row

In [ ]:
import time

# player1 green minimizing player
# player2 red  maximizing player

ROWS = 6
COLS = 7

board = []
windows = []
weights = [1, 10, 100]

for row in range(ROWS):
    board.append([])
    for col in range(COLS):
        board[row].append(0)

In [ ]:
def create_windows():
    windows.clear() #so that a new game restarts from fresh board

    # horizontal: 24
    for row in range(ROWS):
        for col in range(COLS - 3):
            windows.append([(row, col + k) for k in range(4)])

    # vertical: 21
    for row in range(ROWS - 3):
        for col in range(COLS):
            windows.append([(row + k, col) for k in range(4)])

    # diagonal down-right: 12
    for row in range(ROWS - 3):
        for col in range(COLS - 3):
            windows.append([(row + k, col + k) for k in range(4)])

    # diagonal down-left: 12
    for row in range(ROWS - 3):
        for col in range(3, COLS):
            windows.append([(row + k, col - k) for k in range(4)])

In [ ]:
def evaluate(board):
    score = 0
    for w in windows:
        cells = [board[r][c] for r, c in w]
        r_count = cells.count("R")
        g_count = cells.count("G")

        if r_count > 0 and g_count > 0:
            continue

        count = max(r_count, g_count)  # at most one of them is non-zero here
        if count < 1:
            continue

        s = weights[count - 1]

        if count == 3:
            r, c = w[cells.index(0)]  # the single empty cell
            gap = drop_row(board, c) - r  # empty cells below it
            s = s // (1 + gap)

        if g_count > 0:  # sign applied once, at the end
            s = -s

        score += s

    return score

In [ ]:
def check_valid(row, col):
    return 0 <= col < COLS and 0 <= row < ROWS and board[row][col] == 0


def drop_row(board, col):
    # row 0 is the top, so scan from the bottom up
    for r in range(ROWS - 1, -1, -1):
        if board[r][col] == 0:
            return r
    return None


def is_full(board):
    return all(board[0][c] != 0 for c in range(COLS))

In [ ]:
def print_board(board):
    for row in board:
        # print(" ".join("." if x == 0 else x for x in row))
        print(
            " ".join(
                (
                    "."
                    if x == 0
                    else "\033[31mR\033[0m" if x == "R" else "\033[32mG\033[0m"
                )
                for x in row
            )
        )
    print(" ".join(str(c + 1) for c in range(COLS)))


def check_winner(board, player):
    for w in windows:
        val = board[w[0][0]][w[0][1]]
        check = val == player
        for i in range(1, 4):
            if board[w[i][0]][w[i][1]] != val:
                check = False
        if check:
            return True

    return False

In [ ]:
def humanInput(board, symbol):
    print("Human's Turn\nEnter the column value: ")
    try:
        option = int(input()) - 1
        while not check_valid(0, option):
            print(f"Column {option+1} is full or invalid!\nEnter other column value: ")
            option = int(input()) - 1

        board[drop_row(board, option)][option] = symbol
        return True
    except:
        print("Invalid input!!\n")
        return False

In [ ]:
def minimax(board, max_player, depth, alpha, beta, use_pruning):
    if check_winner(board, "R"):
        return 1000000 + depth  # faster win scores higher
    if check_winner(board, "G"):
        return -1000000 - depth
    if is_full(board):
        return 0
    if depth == 0:
        return evaluate(board)

    if max_player:
        best_score = -float("inf")
        for i in [3,2,4,1,5,0,6,]:
            if check_valid(0, i):
                j = drop_row(board, i)
                board[j][i] = "R"
                score = minimax(board, False, depth - 1, alpha, beta, use_pruning)
                board[j][i] = 0
                best_score = max(best_score, score)
                
                if not use_pruning: continue
                alpha = max(alpha, best_score)
                if alpha >= beta:
                    return best_score

        return best_score

    else:
        best_score = float("inf")
        for i in [3,2,4,1,5,0,6,]:
            if check_valid(0, i):
                j = drop_row(board, i)
                board[j][i] = "G"
                score = minimax(board, True, depth - 1, alpha, beta, use_pruning)
                board[j][i] = 0
                best_score = min(best_score, score)
                
                if not use_pruning: continue
                beta = min(beta, best_score)
                if alpha >= beta:
                    return best_score

        return best_score


In [ ]:
def best_move(board, level, symbol, use_pruning = True):
    if level == "Easy":
        depth = 0
    elif level == "Medium":
        depth = 2
    else:
        depth = 4

    if symbol == "R":
        best_score = -float("inf")
    else:
        best_score = float("inf")
    move = None
    alpha = -float("inf")
    beta = float("inf")

    for i in [3,2,4,1,5,0,6,]:  # prioritizing center cols first as they are part of many windows
        if check_valid(0, i):
            j = drop_row(board, i)
            board[j][i] = symbol
            if symbol == "R":
                #red wants the highest score
                score = minimax(board, False, depth, alpha, beta, use_pruning)  #searching till depth based on the chosen level
                if score > best_score:
                    best_score = score
                    move = i
            else:
                #green wants the lowest score
                score = minimax(board, True, depth, alpha, beta, use_pruning)
                if score < best_score:
                    best_score = score
                    move = i
            board[j][i] = 0
    return move

In [ ]:
def main():
    create_windows()

    # total_time_by_pruning = 0
    # total_time_no_pruning = 0

    total_time_pruning = {1: 0, 2: 0}
    total_time_no_pruning = {1: 0, 2: 0}
    

    for r in range(ROWS):
        for c in range(COLS):
            board[r][c] = 0

    player1 = input("Enter whether you want player 1 to be 'Human' or 'Agent':")
    player2 = input("Enter whether you want player 2 to be 'Human' or 'Agent':")

    if player1 not in ("Human", "Agent") or player2 not in ("Human", "Agent"):
        print("Invalid Input!!")
        return

    level_player1 = None
    level_player2 = None

    if player1 == "Agent":
        level_player1 = input("Enter the level of player1(Easy, Medium, Hard):")
        if level_player1 not in ("Easy", "Medium", "Hard"):
            print("Invalid Input!!")
            return

    if player2 == "Agent":
        level_player2 = input("Enter the level of player2(Easy, Medium, Hard):")
        if level_player2 not in ("Easy", "Medium", "Hard"):
            print("Invalid Input!!")
            return

    print("----------Four in a Row Game----------\n")
    print(f"Player1 -> {player1} - G (Green)\nPlayer2 -> {player2} - R (Red)\n")

    current = 1
    while True:
        print_board(board)

        if is_full(board):
            print("It's a Draw!")
            break

        if current == 1:
            player = player1
            level = level_player1
            symbol = "G"
        else:
            player = player2
            level = level_player2
            symbol = "R"

        if player == "Agent":
            print(f"\nPlayer {current} AI ({level}) is thinking ..\n")
            
            b4_move = time.time()
            move = best_move(board, level, symbol)
            total_time_pruning[current] += time.time() - b4_move

            b4_move = time.time()
            best_move(board, level, symbol, use_pruning = False)
            total_time_no_pruning[current] += time.time() - b4_move
            
            board[drop_row(board, move)][move] = symbol
            print(f"Player {current} AI chooses column {move+1}\n")
            time.sleep(1)  # so we can watch the AI moves
        else:
            print(f"Player {current} ({symbol})")
            if humanInput(board, symbol) == False:
                break

        if check_winner(board, symbol):
            print_board(board)
            print(f"Player {current} ({player}) wins!")

            if player1 == "Agent":
                print("\nTotal time taken by player 1:")
                print(f"\tRegular Minimax: {total_time_no_pruning[1]:.3f}")
                print(f"\tWith alpha-beta pruning: {total_time_pruning[1]:.3f}")
            
            if player2 == "Agent":
                print("\nTotal time taken by player 2:")
                print(f"\tRegular Minimax: {total_time_no_pruning[2]:.3f}")
                print(f"\tWith alpha-beta pruning: {total_time_pruning[2]:.3f}")
                
        
            break

        if current == 1:
            current = 2
        else:
            current = 1



main()